In [1]:
import os
from pathlib import Path

import geopandas as gpd
import numpy as np
import requests
from shapely.geometry import Polygon

from shadowroutes.config import settings

src_path = Path('../')
os.chdir(src_path)

## Parameters

In [2]:
# Data paths

MIN_CON_URL = "https://srvprodarcgisp.economia.gob.mx/server/rest/services/Hosted/CartoWebMap_Pro4_WFL1/FeatureServer/16/query"

# Mining concessions (Ministry of Economy) - obtained in November 2025
min_con_path = Path(settings.DATA_ROOT / 'secretaria_economia' / 'raw' / '2025_mining_concessions')
min_con_proc_path = Path(settings.DATA_ROOT / 'secretaria_economia' / 'preprocessed' / '2025_mining_concessions_panel')

# Geodata (Marco Geoestadístico 2024 INEGI) - obtained in November 2025
states_shp_path = Path(settings.DATA_ROOT / 'marco_geoestadistico_inegi' / 'raw' / 'mg_2024_integrado' / '00ent.shp')
mun_shp_path = Path(settings.DATA_ROOT / 'marco_geoestadistico_inegi' / 'raw' / 'mg_2024_integrado' / '00mun.shp')

# Output path
out_mining_panel = Path(settings.DATA_ROOT / 'processed' / 'mining_index_panel.csv')


## Data

### Get mining concessions data (2025)

In [3]:
def fetch_concesiones_gdf(max_chunk=2000):
    all_attrs = []
    all_geoms = []
    offset = 0

    while True:
        params = {
            "where": "1=1",
            "outFields": "*",
            "returnGeometry": "true",
            "f": "json",  # ESRI JSON, not geojson
            "resultOffset": offset,
            "resultRecordCount": max_chunk,
        }
        r = requests.get(MIN_CON_URL, params=params)
        r.raise_for_status()
        data = r.json()

        if "error" in data:
            raise RuntimeError(f"ArcGIS error: {data['error']}")

        features = data.get("features", [])
        if not features:
            break

        for f in features:
            attrs = f.get("attributes", {}) or {}
            geom_dict = f.get("geometry", {}) or {}

            rings = geom_dict.get("rings")
            if rings:
                # Take first ring as main polygon
                poly = Polygon(rings[0])
            else:
                poly = None

            all_attrs.append(attrs)
            all_geoms.append(poly)

        offset += len(features)
        print(f"Fetched {len(features)} records, total so far: {len(all_attrs)}")

        if len(features) < max_chunk:
            break

    # Build GeoDataFrame
    gdf = gpd.GeoDataFrame(all_attrs, geometry=all_geoms)

    # This is Web Mercator
    gdf = gdf.set_crs("EPSG:3857").to_crs("EPSG:4326")

    return gdf


In [4]:
df_min_con = fetch_concesiones_gdf()
print(df_min_con.shape)
df_min_con.head()

Fetched 2000 records, total so far: 2000
Fetched 2000 records, total so far: 4000
Fetched 2000 records, total so far: 6000
Fetched 2000 records, total so far: 8000
Fetched 2000 records, total so far: 10000
Fetched 2000 records, total so far: 12000
Fetched 2000 records, total so far: 14000
Fetched 2000 records, total so far: 16000
Fetched 2000 records, total so far: 18000
Fetched 2000 records, total so far: 20000
Fetched 2000 records, total so far: 22000
Fetched 129 records, total so far: 22129
(22129, 13)


,fid,fecha_sol,estado,expedicion,nombrelote,superficie,expediente,municipio,titulo,SHAPE__Length,titular_ho,SHAPE__Area,geometry
0,1,-2.209162e+12,Nuevo León,-2209161600000,AGUA DULCE NUEVE,100.00,,García,164263,4452.021548,"INDUSTRIA DEL ALCALI, S.A. ADOPTO LA MODALIDAD...",1.238762e+06,"POLYGON ((-100.60347 25.79521, -100.596 25.789..."
1,2,1.193789e+12,Sinaloa,1389312000000,OLIVIA,9783.5763,,Sinaloa,242727,51258.748366,JESUS MANUEL QUINTANA GONZALEZ,1.340590e+08,"POLYGON ((-107.72808 25.96786, -107.80797 25.9..."
2,3,9.747648e+11,Hidalgo,1001980800000,LA PURISIMA 2,26,,Jacala de Ledezma,214540,2790.197831,JUAN EDUARDO GOSCH PATI O,2.995620e+05,"POLYGON ((-99.19976 20.9365, -99.20265 20.9365..."
3,4,1.309478e+12,Mlxico,1359504000000,MAYAL,2166.0957,,Tlatlaya,241635,34561.109597,PABLO ARZATE RODRIGUEZ 35 | JOEL JAIMES DE NOV...,2.423648e+07,"POLYGON ((-100.37317 18.54585, -100.37317 18.5..."
4,5,1.370390e+12,Sonora,1415059200000,SHU YANG,6666.0206,,Aconchi,243719,50756.500186,NEW BEST INTERNATIONAL MINING INVESTMENT GROUP...,9.018662e+07,"POLYGON ((-110.24706 29.87528, -110.24533 29.8..."


In [ ]:
# Save unprocessed output in different formats
df_min_con.to_file(min_con_path.with_suffix('.shp'))
df_min_con.to_file(min_con_path.with_suffix('.geojson'), driver="GeoJSON")
df_min_con.to_csv(min_con_path.with_suffix('.csv'), index=False)


### Get municipalities data (Marco Geoestadístico 2024)

In [6]:
gdf_munis = gpd.read_file(mun_shp_path)
gdf_munis = gdf_munis.to_crs("EPSG:4326")
gdf_munis.head(10)

,CVEGEO,CVE_ENT,CVE_MUN,NOMGEO,geometry
0,01005,01,005,Jesús María,"POLYGON ((-102.35391 22.06255, -102.35312 22.0..."
1,01008,01,008,San José de Gracia,"POLYGON ((-102.45537 22.31212, -102.45501 22.3..."
2,01001,01,001,Aguascalientes,"POLYGON ((-102.10732 22.07476, -102.10699 22.0..."
3,01004,01,004,Cosío,"POLYGON ((-102.29739 22.45527, -102.29554 22.4..."
4,01011,01,011,San Francisco de los Romo,"POLYGON ((-102.15938 22.099, -102.15638 22.097..."
5,01006,01,006,Pabellón de Arteaga,"POLYGON ((-102.25345 22.18302, -102.2513 22.18..."
6,01003,01,003,Calvillo,"POLYGON ((-102.68534 22.10558, -102.6847 22.10..."
7,01010,01,010,El Llano,"POLYGON ((-102.03427 22.06936, -102.03222 22.0..."
8,01002,01,002,Asientos,"POLYGON ((-102.0629 22.30175, -102.06257 22.30..."
9,01009,01,009,Tepezalá,"POLYGON ((-102.17738 22.36243, -102.17968 22.3..."


In [8]:
# Filter to the three states
target_states = ["12", "16", "15"]  # Guerrero, Michoacán, Estado de México
gdf_munis_sub = gdf_munis[gdf_munis["CVE_ENT"].isin(target_states)].copy()

In [9]:
print(gdf_munis_sub.shape)

(323, 5)


### Merge datasets

In [10]:
# Make sure both are in WGS84
df_min_con  = df_min_con.to_crs("EPSG:4326")
gdf_munis_sub = gdf_munis_sub.to_crs("EPSG:4326")

# Spatial join: which concession falls in which municipality
gdf_min_con = gpd.sjoin(gdf_munis_sub, df_min_con, how="left", predicate="intersects")
display(gdf_min_con.head())

# Aggregate: concessions per municipality
gdf_min_con_idx = (
    gdf_min_con.groupby(["CVE_ENT", "CVEGEO", "NOMGEO"])["fid"]  # or any field from concessions
          .count()
          .reset_index(name="num_concessions")
)

# Build the mining index
gdf_min_con_idx["has_mining"] = (gdf_min_con_idx["num_concessions"] > 0).astype(int)
gdf_min_con_idx["concessions_log"] = np.log1p(gdf_min_con_idx["num_concessions"])
gdf_min_con_idx["concessions_scaled"] = (
    (gdf_min_con_idx["concessions_log"] - gdf_min_con_idx["concessions_log"].min()) /
    (gdf_min_con_idx["concessions_log"].max() - gdf_min_con_idx["concessions_log"].min())
)
gdf_min_con_idx["mining_idx"] = 0.5 * gdf_min_con_idx["has_mining"] + 0.5 * gdf_min_con_idx["concessions_scaled"]

print(gdf_min_con_idx.shape)
gdf_min_con_idx.head()

,CVEGEO,CVE_ENT,CVE_MUN,NOMGEO,geometry,index_right,fid,fecha_sol,estado,expedicion,nombrelote,superficie,expediente,municipio,titulo,SHAPE__Length,titular_ho,SHAPE__Area
376,12033,12,033,Huamuxtitlán,"POLYGON ((-98.47244 17.97081, -98.47298 17.969...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
377,12031,12,031,General Canuto A. Neri,"POLYGON ((-100.07588 18.54483, -100.07398 18.5...",3036.0,3037.0,1.171411e+12,Guerrero,1.182470e+12,MIMI,250,,General Canuto A. Neri,229852,7387.193006,NORBERTO FERNANDEZ SAMANO 50 | MAXIMINO RODRIG...,2.791478e+06
377,12031,12,031,General Canuto A. Neri,"POLYGON ((-100.07588 18.54483, -100.07398 18.5...",12573.0,12574.0,1.341878e+12,Guerrero,1.412294e+12,JEHOVA 2,200,,Arcelia,243445,8660.120418,RAFAEL PRESTEGUI ALVARADO 50 | ANADELIA SANCHE...,4.467406e+06
377,12031,12,031,General Canuto A. Neri,"POLYGON ((-100.07588 18.54483, -100.07398 18.5...",10895.0,10896.0,1.265846e+12,Guerrero,1.279238e+12,EL CANDIL,200,,Arcelia,236570,6334.491407,ROGELIO SAUL REYES CERDA 66 | JOSE LUIS CARRIL...,2.233707e+06
377,12031,12,031,General Canuto A. Neri,"POLYGON ((-100.07588 18.54483, -100.07398 18.5...",13558.0,13559.0,1.333411e+12,Guerrero,1.365552e+12,LA CONCEPCIAN,200,,Arcelia,241966,6335.392108,RAFAEL PRESTEGUI ALVARADO 50 | ANADELIA SANCHE...,2.234336e+06


(323, 8)


,CVE_ENT,CVEGEO,NOMGEO,num_concessions,has_mining,concessions_log,concessions_scaled,mining_idx
0,12,12001,Acapulco de Juárez,6,1,1.945910,0.431383,0.715692
1,12,12002,Ahuacuotzingo,0,0,0.000000,0.000000,0.000000
2,12,12003,Ajuchitlán del Progreso,4,1,1.609438,0.356792,0.678396
3,12,12004,Alcozauca de Guerrero,0,0,0.000000,0.000000,0.000000
4,12,12005,Alpoyeca,0,0,0.000000,0.000000,0.000000


In [11]:
print(gdf_min_con_idx["num_concessions"].describe())
print(gdf_min_con_idx["mining_idx"].min(), gdf_min_con_idx["mining_idx"].max())
gdf_min_con_idx.has_mining.value_counts()

count    323.000000
mean       5.349845
std       12.911965
min        0.000000
25%        0.000000
50%        0.000000
75%        4.000000
max       90.000000
Name: num_concessions, dtype: float64
0.0 1.0


has_mining
0    185
1    138
Name: count, dtype: int64

In [12]:
# Format columns
gdf_min_con_idx.columns = gdf_min_con_idx.columns.str.lower()
gdf_min_con_idx.rename(columns={'cve_ent': 'state_id', 'cvegeo': 'muni_id', 'nomgeo': 'municipality'}, inplace=True)
gdf_min_con_idx.head(10)

,state_id,muni_id,municipality,num_concessions,has_mining,concessions_log,concessions_scaled,mining_idx
0,12,12001,Acapulco de Juárez,6,1,1.945910,0.431383,0.715692
1,12,12002,Ahuacuotzingo,0,0,0.000000,0.000000,0.000000
2,12,12003,Ajuchitlán del Progreso,4,1,1.609438,0.356792,0.678396
3,12,12004,Alcozauca de Guerrero,0,0,0.000000,0.000000,0.000000
4,12,12005,Alpoyeca,0,0,0.000000,0.000000,0.000000
5,12,12006,Apaxtla,12,1,2.564949,0.568617,0.784308
6,12,12007,Arcelia,21,1,3.091042,0.685245,0.842622
7,12,12008,Atenango del Río,3,1,1.386294,0.307324,0.653662
8,12,12009,Atlamajalcingo del Monte,0,0,0.000000,0.000000,0.000000
9,12,12010,Atlixtac,0,0,0.000000,0.000000,0.000000


### Save panel data

In [13]:
gdf_min_con_idx.to_csv(out_mining_panel, index=False)